In [1]:
import pandas as pd
import numpy as np
from pyMyriad import *

In [2]:
df = pd.DataFrame({
  "id": np.arange(1000),
  "Gender": np.random.choice(["M", "F"], 1000),
  "Age": np.random.randint(18, 70, 1000),
  "Income": np.random.normal(50000, 15000, 1000),
  "Country": np.random.choice(["Benin", "Albania"], 1000),
  "A": np.random.normal(0, 1, 1000),
  "B": np.random.normal(0, 10, 1000)
})

df.head()

,id,Gender,Age,Income,Country,A,B
0,0,F,62,56647.200631,Albania,-1.776346,11.436856
1,1,M,27,31282.208669,Benin,1.235134,-6.232825
2,2,F,46,65213.080602,Benin,-1.079442,1.873367
3,3,M,60,41207.243767,Benin,-0.158775,-2.737478
4,4,F,67,67345.167377,Albania,0.652430,-16.243065


# Basic usage
The analysis tree is constructed by successive split and analyis adding nodes to the tree.
- `split_*` methods add split nodes, dividing or filtering the dataframe.
- `analyze_*` methods add an analysis node that will perform the analysis.

Split and analysis can be specified either as sting of functions.
Note: use `df` to refer to the dataframe available at the node.

The resulting analyis tree can be printed to visualize the analysis plan

In [3]:
mfun = lambda df: np.mean(df.Income)
nfun = lambda df: np.std(df.Income)
benin_fun =  lambda df: df.Country == 'Benin'
age_40 = lambda df: df.Age > 40
age_60 = lambda df: df.Age > 60

atree = AnalysisTree()\
  .split_by("df.Gender")\
  .split_by(label = "Benin Y/N", expr = benin_fun)\
  .split_by(label = "Age > 40", age40 = age_40, age60 = age_60)\
  .analyze_by(m = mfun, n = nfun)

print(atree)

Analysis Tree
  └- Split Node df.Gender: [df.Gender]
    └- Split Node Benin Y/N: [benin_fun =  lambda df: df.Country == 'Benin']
      └- Split Node Age > 40: [age40: age_40 = lambda df: df.Age > 40 -- age60: age_60 = lambda df: df.Age > 60]
          └- Analysis Node: 
              m: mfun = lambda df: np.mean(df.Income)
              n: nfun = lambda df: np.std(df.Income)



The `run` method applies the analysis plan on a dataframe and returns a `DataTree`.

In [4]:
res = atree.run(df)
print(res)

Data Tree
  Split df.Gender
    └- F
      Split Benin Y/N
        └- False
          Split Age > 40
            └- age40
              analysis :
              └- m: 49297.51983913714
              └- n: 14186.519575470287
            └- age60
              analysis :
              └- m: 47922.759079586736
              └- n: 12635.830142095427
        └- True
          Split Age > 40
            └- age40
              analysis :
              └- m: 47099.759830625524
              └- n: 15026.61136760152
            └- age60
              analysis :
              └- m: 47408.89719733364
              └- n: 12973.372237989199
    └- M
      Split Benin Y/N
        └- False
          Split Age > 40
            └- age40
              analysis :
              └- m: 48734.036048574824
              └- n: 14480.589852677913
            └- age60
              analysis :
              └- m: 49750.8837201623
              └- n: 16944.021272763177
        └- True
          Split Age > 40
     

In [11]:
atree = AnalysisTree(id = "id")\
  .split_by("df.Gender")\
  .split_by(label = "Benin Y/N", expr = benin_fun)\
  .split_by(label = "Age > 40", age40 = age_40, age60 = age_60)\
  .analyze_by(m = mfun, n = nfun)
res = atree.run(df)
print(res._N)
print(res['df.Gender']['F']._N)



1000
555
